# València traffic · Data preparation

This portfolio notebook independently expresses the Hito 2 target calculation and audits the saved analytical dataset. It uses pandas and GeoPandas with a Parquet backend. The target function accepts a locally assembled hourly panel with columns `gid`, `fecha_hora` and `Lectura`. Timestamps should already represent València local time and readings should be numeric. The function is supplied for reuse when that panel is available.

The original raw-file loader, segment parsing and spatial feature cells extend instructor-provided code. Those cells and the course support package are omitted. The [data preparation notes](../docs/02_data_preparation.md) describe the spatial operations, and the [data notes](../data/README.md) document input availability.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

## Construct the afternoon target

Filter to Monday–Friday at hours 17–19 in March 2025 and exclude Fallas (15–19 March inclusive). Check that the selected panel has all 54 eligible timestamps for every segment. Count nonnegative readings as valid and retain the per-segment mean where at least half the records are valid. This function independently implements the study definition without any course loader.

In [ ]:
def build_afternoon_target(panel):
    timestamp = pd.to_datetime(panel["fecha_hora"])
    eligible_window = (
        timestamp.dt.year.eq(2025)
        & timestamp.dt.month.eq(3)
        & timestamp.dt.dayofweek.lt(5)
        & timestamp.dt.hour.isin((17, 18, 19))
        & ~timestamp.dt.day.between(15, 19)
    )
    eligible = panel.loc[eligible_window, ["gid", "Lectura"]].assign(
        hour=timestamp.loc[eligible_window].dt.floor("h")
    )
    if eligible["gid"].isna().any() or eligible.duplicated(["gid", "hour"]).any():
        raise ValueError("The eligible panel must have a unique segment key per hour.")

    total = eligible.groupby("gid").size()
    if eligible["hour"].nunique() != 54 or not total.eq(54).all():
        raise ValueError("Expected 54 eligible hours for every segment.")

    valid = eligible.loc[eligible["Lectura"].ge(0)].groupby("gid")["Lectura"]
    summary = pd.DataFrame({
        "intensidad": valid.mean(),
        "n_horas_validas": valid.count().reindex(total.index, fill_value=0),
        "n_horas_totales": total,
    })
    summary["pct_horas_validas"] = 100 * summary["n_horas_validas"] / total
    return summary.loc[summary["pct_horas_validas"].ge(50)].reset_index()

## Verify the saved GeoParquet

Run this section with a local copy of the Hito 2 GeoParquet. The path works from the project root or the notebook directory. The input is optional so the notebook can also be opened and run from a public clone. The audit prints aggregate values only and does not write a dataset.

In [ ]:
PREDICTORS = [
    "longitud_tramo_m", "superficie_edificada_m2", "n_edificios",
    "prop_edificada", "dens_edificios_km2", "long_via_principal",
    "long_via_secundaria", "long_via_local", "prop_via_principal",
    "n_transporte", "dist_estacion_m", "poblacion_estimada",
    "densidad_poblacion_km2",
]
COVERAGE_FIELDS = ["n_horas_validas", "n_horas_totales", "pct_horas_validas"]


def audit_dataset(segments):
    assert segments.shape == (392, 22)
    assert segments["gid"].notna().all() and segments["gid"].is_unique
    assert segments.crs is not None and segments.crs.to_epsg() == 25830
    assert segments.geometry.notna().all()
    assert segments.geometry.is_valid.all() and not segments.geometry.is_empty.any()

    observed = segments["intensidad"].notna()
    assert observed.sum() == 353 and (~observed).sum() == 39
    assert segments["tiene_objetivo"].eq(observed).all()
    assert segments.loc[observed, "intensidad"].ge(0).all()
    assert segments.loc[observed, "n_horas_totales"].eq(54).all()
    assert segments.loc[observed, "n_horas_validas"].between(27, 54).all()
    assert segments.loc[observed, "pct_horas_validas"].between(50, 100).all()
    assert segments.loc[~observed, COVERAGE_FIELDS].isna().all().all()
    assert segments[PREDICTORS].notna().all().all()

    return {
        "Spatial rows": len(segments),
        "Columns": segments.shape[1],
        "Target-bearing rows": int(observed.sum()),
        "Application rows": int((~observed).sum()),
        "Median target (veh/h)": round(segments["intensidad"].median(), 2),
        "Mean target (veh/h)": round(segments["intensidad"].mean(), 2),
        "Candidate predictors with missing values": int(
            segments[PREDICTORS].isna().any().sum()
        ),
    }


project = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
dataset_path = project / "data" / "processed" / "dataset_T_r400_tarde_u50.parquet"
if dataset_path.is_file():
    for label, value in audit_dataset(gpd.read_parquet(dataset_path)).items():
        print(f"{label}: {value}")
else:
    print("Local Hito 2 GeoParquet unavailable. See the verified summary below.")

## Verified findings

The saved dataset contains **392 rows × 22 columns**, including **353 target-bearing and 39 application segments**. The retained spatial targets have a median of **677.04 veh/h** and a mean of **1,270.33 veh/h**. All 13 candidate predictors are complete. The local control export records **27 passes**.

The assignment gives a **677 veh/h reference median**. The working notebook records a **691 veh/h median before the geometry join**, when there were 355 qualifying targets. These are target summaries. Model fitting, validation and mapping are future work.